<a href="https://colab.research.google.com/github/pendar-hadinezhad/Khayyam-Language-Model-Persian/blob/main/KhayyamLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Building a Persian Language Model (KhayyamLLM) from scratch**

Data Preprocessing

In [ ]:
with open ("persian_poems.txt" , "r" , encoding = "utf_8") as f:
  raw_text = f.read()

In [ ]:
print(f"the total number of charachters : {len(raw_text)}")

the total number of charachters : 6111676


In [ ]:
print(raw_text[:200])


  	
به نام خداوند جان و خرد
کزین برتر اندیشه برنگذرد
خداوند نام و خداوند جای
خداوند روزی ده رهنمای
خداوند کیوان و گردان سپهر
فروزنده ماه و ناهید و مهر
ز نام و نشان و گمان برترست
نگارنده بر شده پیکرست


In [ ]:
pip install hazm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.6 MB/s eta 0:00:00
  Created wheel for flashtext: filename=flashtext-2.7-py2.py3-none-any.whl size=9371 sha256=417891e69838267696b95141551547a6e89fc20806c9b54a81769ef1698c21b2
  Stored in directory: /root/.cache/pip/wheels/be/12/c9/228313ff5cb722777830302f1d4136de4129932e6a0354c056
Successfully built flashtext


In [ ]:
import re
import unicodedata

from hazm import Normalizer
text = unicodedata.normalize("NFKC", raw_text)

KeyboardInterrupt: 

In [ ]:
def clean_persian_text(text):

    # ---------------------------------------
    # 1. Unicode normalization
    # ---------------------------------------
# این یکی از مهم‌ترین قسمت‌های کد است.

# Unicode روش استانداردی برای نمایش کاراکترها در کامپیوتر است.

# گاهی یک حرف ممکن است با چند representation مختلف ذخیره شده باشد.

# NFKC تلاش می‌کند representationهای سازگار را به یک فرم استاندارد تبدیل کند.

# مثلاً بعضی کاراکترهای مخصوص PDF:

# ﻛ

# ممکن است به فرم استاندارد نزدیک شوند.

# 5. چرا Unicode برای LLM مهم است؟

# فرض کن در دیتاست این دو را داریم:

# کتاب
# كﺘاب

# انسان می‌گوید:

# هر دو «کتاب» هستند.

# ولی tokenizer ممکن است آن‌ها را متفاوت ببیند.

# مثلاً:

# کتاب
# ↓
# ["کت", "اب"]

# اما:

# كﺘاب
# ↓
# ["ك", "ﺘ", "اب"]

# پس یک کلمه، vocabulary بیشتری مصرف می‌کند.

    text = unicodedata.normalize("NFKC", text)

    # ---------------------------------------
    # 2. Remove PDF/control characters
    # ---------------------------------------

    # Form feed, vertical tab, etc.
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)

    # ---------------------------------------
    # 3. Persian/Arabic character normalization
    # ---------------------------------------

    replacements = {
        "ي": "ی",
        "ى": "ی",
        "ئ": "ی",
        "ك": "ک",
        "ة": "ه",
        "ۀ": "ه",
        "ؤ": "و",
        "إ": "ا",
        "أ": "ا",
        "ٱ": "ا",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # ---------------------------------------
    # 4. Remove Arabic/Persian diacritics
    # ---------------------------------------

    # Harakat
    diacritics = re.compile(
        r"[\u064B-\u065F\u0670\u06D6-\u06ED]"
    )

    text = diacritics.sub("", text)
# """
#     # ---------------------------------------
#     # 5. Remove tatweel / kashida بنابراین:

# ﻣــــﻦ#

# می‌تواند تبدیل شود به#:

# من#

# این قسمت برای دیتاست PDF شما خیلی مهم است.
#     # ---------------------------------------"""

    text = text.replace("ـ", "")

    # ---------------------------------------
    # 6. Remove unusual presentation characters
    # ---------------------------------------

    # Anything that remains in Arabic
    # Presentation Forms blocks
    #بعضی PDFها حروف عربی/فارسی را به شکل Arabic Presentation Forms ذخیره می‌کنند.

# مثل:

# ﻛ
# ﻲ
# ﻣ
    text = re.sub(
        r"[\uFB50-\uFDFF\uFE70-\uFEFF]",
        "",
        text
    )

     # =========================================
    # 7. REMOVE ENGLISH
    # =========================================

    # English uppercase/lowercase
    text = re.sub(
        r"[A-Za-z]",
        "",
        text
    )


    # ---------------------------------------
    # 7. Normalize ZWNJ
    # ---------------------------------------

    # Keep ZWNJ because it is useful in Persian.
    # Convert multiple ZWNJs to one.

#     14. چرا نیم‌فاصله را حذف نمی‌کنیم؟

# این خیلی مهم است.

# مثلاً:

# می‌روم

# با:

# میروم

# از نظر NLP یکسان نیستند.

# برای یک tokenizer فارسی، نیم‌فاصله می‌تواند اطلاعات مفیدی داشته باشد.

# بنابراین در کد:
    text = re.sub(
        r"\u200c+",
        "\u200c",
        text
    )

    # Remove ZWNJ next to whitespace
    text = re.sub(
        r"\s+\u200c|\u200c\s+",
        " ",
        text
    )

# چه کاراکترهایی اجازه دارند در دیتاست باقی بمانند؟
    allowed_pattern = (
    r"[^\u0600-\u06FF"
    r"\u200C"
    r"۰-۹"
    # r"0-9"
    r"\s"
    r"\.\,\!\؟\؛\:\،\;"
    r"\(\)\[\]\{\}"
    r"\-"
    r"«»"
    r"ـ]"
)

    re.sub(allowed_pattern, " ", text)

    # ---------------------------------------
    # 8. Normalize whitespace
    # ---------------------------------------

    text = re.sub(r"[ \t]+", " ", text)

    # Normalize newlines
    text = re.sub(r"\n+", "\n", text)

    # Remove spaces at beginning/end of lines
    text = re.sub(
        r"[ \t]+\n",
        "\n",
        text
    )

    text = re.sub(
        r"\n[ \t]+",
        "\n",
        text
    )

    # ---------------------------------------
    # 9. Hazm normalization
    # ---------------------------------------
# یک Normalizer از Hazm می‌سازیم.

# correct_spacing=True

# Hazm تلاش می‌کند spacing فارسی را اصلاح کند.

# مثلاً بعضی فاصله‌های نامناسب.

    normalizer = Normalizer(
        correct_spacing=True,
        remove_diacritics=True,
        persian_numbers=False
    )

    text = normalizer.normalize(text)

    # ---------------------------------------
    # 10. Final whitespace cleanup
    # ---------------------------------------
# تمیز کردن Space
    text = re.sub(r"[ \t]+", " ", text)
# حذف Space اطراف newline
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()



In [ ]:
import re
import unicodedata
from hazm import Normalizer


# ============================================================
# 1. Compile regex patterns ONCE
# ============================================================

CONTROL_CHARS_RE = re.compile(
    r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]"
)

DIACRITICS_RE = re.compile(
    r"[\u064B-\u065F\u0670\u06D6-\u06ED]"
)

ZWNJ_RE = re.compile(
    r"\u200C+"
)

ZWNJ_SPACE_RE = re.compile(
    r"\s+\u200C|\u200C\s+"
)

ENGLISH_RE = re.compile(
    r"[A-Za-z]+"
)

MULTIPLE_SPACES_RE = re.compile(
    r"[ \t]+"
)

MULTIPLE_NEWLINES_RE = re.compile(
    r"\n{3,}"
)

LINE_START_SPACE_RE = re.compile(
    r"[ \t]+\n"
)

LINE_END_SPACE_RE = re.compile(
    r"\n[ \t]+"
)


# ============================================================
# 2. Persian / Arabic character mapping
# ============================================================

PERSIAN_TRANSLATION = str.maketrans({

    # Arabic Yeh → Persian Yeh
    "ي": "ی",
    "ى": "ی",

    # Arabic Kaf → Persian Kaf
    "ك": "ک",

    # Arabic Heh variants
    "ة": "ه",
    "ۀ": "ه",

    # Alef variants
    "إ": "ا",
    "أ": "ا",
    "ٱ": "ا",

})


# ============================================================
# 3. Allowed characters
# ============================================================

ALLOWED_CHARS_RE = re.compile(
    r"[^\u0600-\u06FF"
    r"\u200C"
    r"۰-۹"
    r"\s"
    r"\.,!؟؛:،;"
    r"\(\)\[\]\{\}"
    r"\-"
    r"«»"
    r"ـ"
    r"]"
)


# ============================================================
# 4. Hazm Normalizer
# ============================================================

normalizer = Normalizer(
    correct_spacing=True,
    remove_diacritics=True,
    persian_numbers=False
)


# ============================================================
# 5. Main function
# ============================================================

def clean_persian_text_v2(text):

    # --------------------------------------------------------
    # Safety
    # --------------------------------------------------------

    if not isinstance(text, str):
        return ""

    # --------------------------------------------------------
    # Unicode normalization
    # --------------------------------------------------------

    text = unicodedata.normalize("NFKC", text)

    # --------------------------------------------------------
    # Remove control characters
    # --------------------------------------------------------

    text = CONTROL_CHARS_RE.sub(" ", text)

    # --------------------------------------------------------
    # Normalize Arabic/Persian characters
    # --------------------------------------------------------

    text = text.translate(PERSIAN_TRANSLATION)

    # --------------------------------------------------------
    # Remove Arabic/Persian diacritics
    # --------------------------------------------------------

    text = DIACRITICS_RE.sub("", text)

    # --------------------------------------------------------
    # Remove Tatweel / Kashida
    # --------------------------------------------------------

    text = text.replace("ـ", "")

    # --------------------------------------------------------
    # Remove English
    # --------------------------------------------------------

    text = ENGLISH_RE.sub("", text)

    # --------------------------------------------------------
    # Normalize ZWNJ
    # --------------------------------------------------------

    text = ZWNJ_RE.sub("\u200C", text)

    # Remove ZWNJ next to whitespace
    text = ZWNJ_SPACE_RE.sub(" ", text)

    # --------------------------------------------------------
    # Remove unwanted characters
    # --------------------------------------------------------

    text = ALLOWED_CHARS_RE.sub(" ", text)

    # --------------------------------------------------------
    # Normalize spaces
    # --------------------------------------------------------

    text = MULTIPLE_SPACES_RE.sub(" ", text)

    # --------------------------------------------------------
    # Normalize newlines
    # --------------------------------------------------------

    text = MULTIPLE_NEWLINES_RE.sub("\n\n", text)

    text = LINE_START_SPACE_RE.sub("\n", text)
    text = LINE_END_SPACE_RE.sub("\n", text)

    # --------------------------------------------------------
    # Hazm normalization
    # --------------------------------------------------------

    text = normalizer.normalize(text)

    # --------------------------------------------------------
    # Final cleanup
    # --------------------------------------------------------

    text = MULTIPLE_SPACES_RE.sub(" ", text)

    text = MULTIPLE_NEWLINES_RE.sub("\n\n", text)

    return text.strip()

In [ ]:
cleaned_text = clean_persian_text_v2(raw_text)
print("Cleaning completed.")

In [ ]:
print("Original characters:", len(raw_text))
print("Clean characters:", len(cleaned_text))

print("\nSample:")
print(cleaned_text[0:200])

In [ ]:
cleaned_text_edited = cleaned_text[50:]

In [ ]:
print(cleaned_text_edited)

In [ ]:
english = re.findall(
    r"[A-Za-z]+",
    cleaned_text
)

print("English words remaining:", len(english))

if english:
    print("Examples:")
    print(english[:50])
else:
    print("No English text found.")

English words remaining: 0
No English text found.


In [ ]:
with open(
    "khayyam_clean.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(cleaned_text_edited)

Generateing token Ids

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Sequence, NFKC
from tokenizers.decoders import BPEDecoder


# ============================================
# Configuration
# ============================================

DATA_FILE = "persian_poems.txt"
TOKENIZER_FILE = "persian_bpe_tokenizer.json"

VOCAB_SIZE = 1000


# ============================================
# Create BPE tokenizer
# ============================================

tokenizer = Tokenizer(
    BPE(
        unk_token="<unk>"
    )
)


# ============================================
# Normalization
# ============================================

# tokenizer.normalizer = Sequence([
#     NFKC()
# ])


# ============================================
# Pre-tokenization
# ============================================

# tokenizer.pre_tokenizer = Whitespace()


# ============================================
# Trainer
# ============================================

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,

    special_tokens=[
        "<pad>",
        "<unk>",
        "<bos>",
        "<eos>"
    ],

    min_frequency=2,

    show_progress=True
)


# ============================================
# Train
# ============================================

tokenizer.train(
    files=[DATA_FILE],
    trainer=trainer
)


# ============================================
# Decoder
# ============================================

tokenizer.decoder = BPEDecoder()


# ============================================
# Save
# ============================================

tokenizer.save(TOKENIZER_FILE)

print("Tokenizer trained successfully.")
print(f"Saved to: {TOKENIZER_FILE}")
print(f"Vocabulary size: {tokenizer.get_vocab_size()}")

Tokenizer trained successfully.
Saved to: persian_bpe_tokenizer.json
Vocabulary size: 1000


In [ ]:
# ============================================
# Test the Tokenizer
# ============================================

# Load the tokenizer back from the file we just saved
loaded_tokenizer = Tokenizer.from_file(TOKENIZER_FILE)

# A test sentence from Khayyam
test_text = "چون عمر به سر رسید ای وای بیا"
encoded = loaded_tokenizer.encode(test_text)
decoded = loaded_tokenizer.decode(encoded.ids)

print(f"\n--- Test Results ---")
print(f"Original: {test_text}")
print(f"Tokens: {encoded.tokens}")
print(f"IDs: {encoded.ids}")
print(f"Decoded: {decoded}")


--- Test Results ---
Original: چون عمر به سر رسید ای وای بیا
Tokens: ['چون ', 'عمر ', 'به ', 'سر ', 'رس', 'ید ', 'ای ', 'و', 'ای ', 'بیا']
IDs: [129, 934, 62, 140, 449, 123, 84, 36, 84, 871]
Decoded: چون عمر به سر رسید ای وای بیا


**DATA SAMPLING**

In [ ]:
with open("persian_poems.txt", "r", encoding="utf-8") as f:
   raw_text = f.read()
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

2555931


In [ ]:
print(enc_text)

Encoding(num_tokens=2555931, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])


In [ ]:
enc_text.tokens

['\n',
 ' ',
 ' ',
 '\t',
 '\n',
 'به ',
 'نام ',
 'خدا',
 'وند ',
 'جان ',
 'و ',
 'خر',
 'د',
 '\n',
 'ک',
 'زین ',
 'بر',
 'تر ',
 'اندیشه ',
 'بر',
 'نگ',
 'ذ',
 'ر',
 'د',
 '\n',
 'خدا',
 'وند ',
 'نا',
 'م و ',
 'خدا',
 'وند ',
 'جای',
 '\n',
 'خدا',
 'وند ',
 'روز',
 'ی ',
 'ده ',
 'ره',
 'نم',
 'ای',
 '\n',
 'خدا',
 'وند ',
 'کی',
 'وان ',
 'و ',
 'گردان ',
 'سپ',
 'هر',
 '\n',
 'ف',
 'روز',
 'نده ',
 'ماه ',
 'و ',
 'ن',
 'اه',
 'ید ',
 'و م',
 'هر',
 '\n',
 'ز ',
 'نا',
 'م و ',
 'نش',
 'ان و ',
 'گ',
 'مان ',
 'بر',
 'تر',
 'ست',
 '\n',
 'نگ',
 'ار',
 'نده ',
 'بر ',
 'شده ',
 'پی',
 'کر',
 'ست',
 '\n',
 'به ',
 'بین',
 'ند',
 'گان ',
 'آفر',
 'ی',
 'ن',
 'نده ',
 'را',
 '\n',
 'ن',
 'بینی ',
 'مر',
 'نج',
 'ان ',
 'دو ',
 'بین',
 'نده ',
 'را',
 '\n',
 'نی',
 'ابد ',
 'بدو ',
 'نیز ',
 'اندیشه ',
 'را',
 'ه',
 '\n',
 'که ',
 'او ',
 'بر',
 'تر ',
 'از ',
 'نا',
 'م و ',
 'از ',
 'جای',
 'گ',
 'اه',
 '\n',
 'سخن ',
 'هر ',
 'چه ',
 'زین ',
 'گو',
 'هر',
 'ان ',
 'بگذ',
 'ر',

In [ ]:
!pip install torch;

In [ ]:
import torch

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt).ids
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=64,
                        stride=32, shuffle=True, drop_last=True,
                         num_workers=0):

      dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
      dataloader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            drop_last=drop_last,
            num_workers=num_workers
            )
      return dataloader

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=128, stride=128, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  5,   7,   7,  ..., 259, 120, 134],
        [ 54, 515,  20,  ...,  35,   5,  62],
        [ 40,  20,  30,  ..., 212, 671,  51],
        ...,
        [701, 214, 124,  ...,  13, 458,  69],
        [747,  13, 173,  ...,  82,  50, 219],
        [ 35, 116, 121,  ..., 307, 488,  76]]), tensor([[  7,   7,   4,  ..., 120, 134,  54],
        [515,  20,  18,  ...,   5,  62,  40],
        [ 20,  30, 354,  ..., 671,  51,  62],
        ...,
        [214, 124, 321,  ..., 458,  69, 747],
        [ 13, 173, 156,  ...,  50, 219,  35],
        [116, 121,  41,  ..., 488,  76,  23]])]


In [ ]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 23, 930,  34,  ..., 122, 165, 287],
        [196,  18,   5,  ..., 550,  18, 163],
        [986, 751,  69,  ...,   5,  62, 752],
        ...,
        [ 95,  33,  61,  ..., 176, 131, 176],
        [187, 441, 217,  ...,  78, 283, 246],
        [ 63,   5, 241,  ...,  62, 187, 811]]), tensor([[930,  34, 182,  ..., 165, 287, 196],
        [ 18,   5,  52,  ...,  18, 163, 986],
        [751,  69,  18,  ...,  62, 752, 161],
        ...,
        [ 33,  61, 111,  ..., 131, 176, 187],
        [441, 217, 777,  ..., 283, 246,  63],
        [  5, 241,  64,  ..., 187, 811, 460]])]


In [ ]:
dataloader = create_dataloader_v1(
 raw_text, batch_size=8, max_length=4, stride=4,
 shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  5,   7,   7,   4],
        [  5,  62, 487, 604],
        [784, 213,  50, 196],
        [ 18,   5,  41, 259],
        [ 69, 325, 705,  69],
        [100,  19,  20,  18],
        [  5, 604, 784, 156],
        [236, 604, 784, 838]])

Targets:
 tensor([[  7,   7,   4,   5],
        [ 62, 487, 604, 784],
        [213,  50, 196,  18],
        [  5,  41, 259,  69],
        [325, 705,  69, 100],
        [ 19,  20,  18,   5],
        [604, 784, 156, 236],
        [604, 784, 838,   5]])


**Creating token embeddings**

In [ ]:
vocab_size = 1000
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

**Positional Encoding**

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
 raw_text, batch_size=8, max_length=max_length,
 stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[  5,   7,   7,   4],
        [  5,  62, 487, 604],
        [784, 213,  50, 196],
        [ 18,   5,  41, 259],
        [ 69, 325, 705,  69],
        [100,  19,  20,  18],
        [  5, 604, 784, 156],
        [236, 604, 784, 838]])

Inputs shape:
 torch.Size([8, 4])


In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [ ]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [ ]:
import torch.nn as nn

# Chapter4:Implementing a GPT model from scratch to generate text

4.1 Coding an LLM architecture

In [ ]:
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))
  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    norm_x = (x - mean) / torch.sqrt(var + self.eps)
    return self.scale * norm_x + self.shift

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`,
        # this will result in errors in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec



In [ ]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh
     (torch.sqrt(torch.tensor(2.0)/torch.pi) * (x + 0.044715 + 0.04*torch.pow(x , 3))
     ))

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
        GELU(),
        nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )
    def forward(self, x):
        return self.layers(x)

In [ ]:
class TransformerBlock(nn.Module):
      def __init__(self, cfg):
            super().__init__()
            self.att = MultiHeadAttention(
                d_in=cfg["emb_dim"],
                d_out=cfg["emb_dim"],
                context_length=cfg["context_length"],
                num_heads=cfg["n_heads"],
                dropout=cfg["drop_rate"],
                qkv_bias=cfg["qkv_bias"])
            self.ff = FeedForward(cfg)
            self.norm1 = LayerNorm(cfg["emb_dim"])
            self.norm2 = LayerNorm(cfg["emb_dim"])
            self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

      def forward(self, x):
            shortcut = x
            x = self.norm1(x)
            x = self.att(x)
            x = self.drop_shortcut(x)
            x = x + shortcut
            shortcut = x
            x = self.norm2(x)
            x = self.ff(x)
            x = self.drop_shortcut(x)
            x = x + shortcut
            return x

In [ ]:
GPT_CONFIG_124M = {
 "vocab_size": 1000, # Vocabulary size
 "context_length": 128, # Context length
 "emb_dim": 768, # Embedding dimension
 "n_heads": 12, # Number of attention heads
 "n_layers": 12, # Number of layers
 "drop_rate": 0.1, # Dropout rate
 "qkv_bias": False # Query-Key-Value bias
}

In [ ]:
torch.manual_seed(123)
x = torch.rand(2, 4, 768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


***Coding the GPT model***

In [ ]:
class GPTModel(nn.Module):
      def __init__(self, cfg):
          super().__init__()
          self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
          self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
          self.drop_emb = nn.Dropout(cfg["drop_rate"])

          self.trf_blocks = nn.Sequential(
              *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

          self.final_norm = LayerNorm(cfg["emb_dim"])
          self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
          )
      def forward(self, in_idx):
          batch_size, seq_len = in_idx.shape
          tok_embeds = self.tok_emb(in_idx)

          pos_embeds = self.pos_emb(
          torch.arange(seq_len, device=in_idx.device)
          )
          x = tok_embeds + pos_embeds
          x = self.drop_emb(x)
          x = self.trf_blocks(x)
          x = self.final_norm(x)
          logits = self.out_head(x)
          return logits

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "ای دوست بیا تا غم فردا نخوریم"
txt2 = "این کوزه چو من عاشق زاری بود"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[12919,   151,   234, 17550,   107, 30335, 45692, 41486, 17550,   101,
           151,   234, 12919, 17550,   103, 12919, 17550,   118, 25405, 18923,
           223, 26897, 38843, 12919, 18923,   228,   148,   106, 30335, 26897,
           151,   234, 25405],
        [12919,   151,   234, 23338,   220,   150,   102, 30335,   148,   110,
         29519,   220,   150,   228, 30335, 47048, 23338, 17550,   117, 34247,
           112,   149,   224, 17550,   110, 12919, 26897,   151,   234, 17550,
           101, 30335, 38843]])


In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 86,662,656


In [ ]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

Token embedding layer shape: torch.Size([1000, 768])
Output layer shape: torch.Size([1000, 768])


***CHAPTER 5 - Pretraining on unlabeled data***

*5.1 Evaluating generative text models*

5.1.1 Using GPT to generate text

In [ ]:
def generate_text_simple(model, idx,
    max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [ ]:
import torch
from tokenizers import Tokenizer

# Load the custom tokenizer trained earlier
custom_tokenizer = Tokenizer.from_file(TOKENIZER_FILE)

def text_to_token_ids(text, tokenizer):
    # Use .ids for the custom tokenizer's encoding
    encoded = tokenizer.encode(text).ids
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
   flat = token_ids.squeeze(0)
   return tokenizer.decode(flat.tolist())

start_context = "ای دوست بیا تا غم فردا نخوریم"

# Use the custom tokenizer
tokenizer = custom_tokenizer

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 ای دوست بیا تا غم فردا نخوریمب ند
ایه چاره یم
ستان بپرروزاز تو کین 


5.1.2 Calculating the text generation loss

Calculating the training and validation set losses

In [ ]:
total_characters = len(raw_text)
total_tokens = len(tokenizer.encode(raw_text))
print("Characters:", total_characters)
print("Tokens:", total_tokens)

Characters: 6111676
Tokens: 2555931


In [ ]:
train_ratio = 0.90
split_idx = int(train_ratio * len(raw_text))
train_data = raw_text[:split_idx]
val_data = raw_text[split_idx:]

In [ ]:
import torch
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)


val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers = 0
    )

In [ ]:
print("Train loader:")
for x, y in train_loader:
   print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
   print(x.shape, y.shape)

Streaming output truncated to the last 5000 lines.
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torch.Size([2, 128])
torch.Size([2, 128]) torc

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
    logits.flatten(0, 1), target_batch.flatten()
    )
    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
           loss = calc_loss_batch(
                  input_batch, target_batch, model, device
                   )
           total_loss += loss.item()
        else:
             break
    return total_loss / num_batches

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 7.090517291600555
Validation loss: 7.080460469846781


Training an LLM

In [ ]:
def train_model_simple(model, train_loader, val_loader,
                        optimizer, device, num_epochs,
                         eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
                  optimizer.zero_grad()
                  loss = calc_loss_batch(
                         input_batch, target_batch, model, device
                         )
                  loss.backward()
                  optimizer.step()
                  tokens_seen += input_batch.numel()
                  global_step += 1

                  if global_step % eval_freq == 0:
                        train_loss, val_loss = evaluate_model(
                            model, train_loader, val_loader, device, eval_iter)
                        train_losses.append(train_loss)
                        val_losses.append(val_loss)
                        track_tokens_seen.append(tokens_seen)
                        print(f"Ep {epoch+1} (Step {global_step:06d}): "
                        f"Train loss {train_loss:.3f}, "
                        f"Val loss {val_loss:.3f}"
                      )
        generate_and_print_sample(
            model, tokenizer, device, start_context
            )
    return train_losses, val_losses, track_tokens_seen

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
        train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
        val_loader, model, device, num_batches=eval_iter
        )
    model.train()
    return train_loss, val_loss

In [ ]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
        model=model, idx=encoded,
        max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()

In [ ]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text).ids
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
   flat = token_ids.squeeze(0)
   return tokenizer.decode(flat.tolist())



In [ ]:
def generate_text_simple(model, idx,
    max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(
     model.parameters(),
    lr=0.0004, weight_decay=0.1
)
num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="ساقی بیا تا غم فردا نخوریم", tokenizer=tokenizer
)

Ep 1 (Step 000000): Train loss 6.819, Val loss 6.802
Ep 1 (Step 000005): Train loss 6.031, Val loss 6.084
Ep 1 (Step 000010): Train loss 5.998, Val loss 5.989
Ep 1 (Step 000015): Train loss 6.031, Val loss 6.022
Ep 1 (Step 000020): Train loss 5.928, Val loss 5.987
Ep 1 (Step 000025): Train loss 5.992, Val loss 5.908
Ep 1 (Step 000030): Train loss 5.870, Val loss 5.882
Ep 1 (Step 000035): Train loss 5.863, Val loss 5.872
Ep 1 (Step 000040): Train loss 5.817, Val loss 5.859
Ep 1 (Step 000045): Train loss 5.871, Val loss 5.823
Ep 1 (Step 000050): Train loss 5.778, Val loss 5.797
Ep 1 (Step 000055): Train loss 5.808, Val loss 5.759
Ep 1 (Step 000060): Train loss 5.793, Val loss 5.725
Ep 1 (Step 000065): Train loss 5.816, Val loss 5.752
Ep 1 (Step 000070): Train loss 5.782, Val loss 5.720
Ep 1 (Step 000075): Train loss 5.713, Val loss 5.720
Ep 1 (Step 000080): Train loss 5.613, Val loss 5.735
Ep 1 (Step 000085): Train loss 5.678, Val loss 5.628
Ep 1 (Step 000090): Train loss 5.701, Val loss

KeyboardInterrupt: 

import torch

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(
    epochs_seen, val_losses, linestyle="-.", label="Validation loss"
    )
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)
    ax2.set_xlabel("Tokens seen")
    fig.tight_layout()
    plt.show()
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

Decoding strategies to control randomness

In [ ]:
model.to("cpu")
model.eval();

In [ ]:
def generate_text_simple(model, idx,
    max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

5.3.1 Temperature scaling

5.3.2 Top-k sampling

5.3.3 Modifying the text generation function

In [ ]:
def generate(model, idx, max_new_tokens, context_size,
            temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
          idx_cond = idx[:, -context_size:]
          with torch.no_grad():
               logits = model(idx_cond)
          logits = logits[:, -1, :]

          if top_k is not None:
                top_logits, _ = torch.topk(logits, top_k)
                min_val = top_logits[:, -1]
                logits = torch.where(
                logits < min_val,
                torch.tensor(float('-inf')).to(logits.device),
                logits
                )
          if temperature > 0.0:
                logits = logits / temperature
                probs = torch.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples=1)
          else:
                idx_next = torch.argmax(logits, dim=-1, keepdim=True)
          if idx_next == eos_id:
                break
          idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [ ]:
torch.manual_seed(123)
token_ids = generate(
      model=model,
      idx=text_to_token_ids("ساقی بیا", tokenizer),
      max_new_tokens=15,
      context_size=GPT_CONFIG_124M["context_length"],
      top_k=25,
      temperature=1.4
      )
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

5.4 Loading and saving model weights in PyTorch

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth", map_location=device))
model.eval()


In [ ]:
torch.save({
 "model_state_dict": model.state_dict(),
 "optimizer_state_dict": optimizer.state_dict(),
 },
 "model_and_optimizer.pth"
)

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth", map_location=device)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

**Loading pretrained weights from OpenAI**

In [ ]:
pip install tensorflow>=2.15.0 tqdm>=4.66

In [ ]:
import urllib.request
url = (
 "https://raw.githubusercontent.com/rasbt/"
 "LLMs-from-scratch/main/ch05/"
 "01_main-chapter-code/gpt_download.py"
)
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

In [ ]:
from gpt_download import download_and_load_gpt2
settings, params = download_and_load_gpt2(
 model_size="124M", models_dir="gpt2"
)

In [ ]:
print("Settings:", settings)
print("Parameter dictionary keys:", params.keys())

In [ ]:
print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)

In [ ]:
model_configs = {
 "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
 "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
 "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
 "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

In [ ]:
model_name = "gpt2-small (124M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])

In [ ]:
NEW_CONFIG.update({"context_length": 1024})

In [ ]:
NEW_CONFIG.update({"qkv_bias": True})

In [ ]:
gpt = GPTModel(NEW_CONFIG)
gpt.eval()

In [ ]:
def assign(left, right):
 if left.shape != right.shape:
    raise ValueError(f"Shape mismatch. Left: {left.shape}, "
    "Right: {right.shape}"
 )
 return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])


In [ ]:
load_weights_into_gpt(gpt, params)
gpt.to(device)

In [ ]:
# torch.manual_seed(123)
token_ids = generate(
  model=gpt,
  idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
  max_new_tokens=25,
  context_size=NEW_CONFIG["context_length"],
  top_k=50,
  temperature=1.5
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

**Fine-tuning for classification**

In [ ]:
!rm -rf sms_spam_collection
!rm -f sms_spam_collection.zip

In [ ]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(
      url, zip_path, extracted_path, data_file_path):

    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download "
      "and extraction."
       )
        return

        with urllib.request.urlopen(url) as response:
            with open(zip_path, "wb") as out_file:
              out_file.write(response.read())

        with zipfile.ZipFile(zip_path, "r") as zip_ref:
              zip_ref.extractall(extracted_path)

        original_file_path = Path(extracted_path) / "SMSSpamCollection"
        os.rename(original_file_path, data_file_path)
        print(f"File downloaded and saved as {data_file_path}")

In [ ]:
download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

In [ ]:
import pandas as pd

df = pd.read_csv(
  data_file_path, sep="\t", header=None, names=["Label", "Text"]
  )

df


In [ ]:
!ls -R sms_spam_collection

In [ ]:
print(df["Label"].value_counts())

In [ ]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(
    num_spam, random_state=123
    )
    balanced_df = pd.concat([
    ham_subset, df[df["Label"] == "spam"]
    ])
    return balanced_df
balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

In [ ]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

In [ ]:
def random_split(df, train_frac, validation_frac):

    df = df.sample(
    frac=1, random_state=123
    ).reset_index(drop=True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    return train_df, validation_df, test_df
train_df, validation_df, test_df = random_split(
   balanced_df, 0.7, 0.1)


In [ ]:
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

In [ ]:
import torch
from torch.utils.data import Dataset
class SpamDataset(Dataset):
      def __init__(self, csv_file, tokenizer, max_length=None,
          pad_token_id=50256):
          self.data = pd.read_csv(csv_file)

          self.encoded_texts = [
          tokenizer.encode(text) for text in self.data["Text"]
          ]
          if max_length is None:
             self.max_length = self._longest_encoded_length()
          else:
             self.max_length = max_length

          self.encoded_texts = [
          encoded_text[:self.max_length]
          for encoded_text in self.encoded_texts
          ]

          self.encoded_texts = [
          encoded_text + [pad_token_id] *
          (self.max_length - len(encoded_text))
          for encoded_text in self.encoded_texts
          ]

      def __getitem__(self, index):
          encoded = self.encoded_texts[index]
          label = self.data.iloc[index]["Label"]
          return (
          torch.tensor(encoded, dtype=torch.long),
          torch.tensor(label, dtype=torch.long)
          )

      def __len__(self):
          return len(self.data)

      def _longest_encoded_length(self):
          max_length = 0
          for encoded_text in self.encoded_texts:
              encoded_length = len(encoded_text)
              if encoded_length > max_length:
                 max_length = encoded_length
          return max_length

In [ ]:
train_dataset = SpamDataset(
  csv_file="train.csv",
  max_length=None,
  tokenizer=tokenizer
)

In [ ]:
print(train_dataset.max_length)

In [ ]:
val_dataset = SpamDataset(
  csv_file="validation.csv",
  max_length=train_dataset.max_length,
  tokenizer=tokenizer
)
test_dataset = SpamDataset(
  csv_file="test.csv",
  max_length=train_dataset.max_length,
  tokenizer=tokenizer
)